In [2]:
import sys
sys.path.append('../')
from src.core.scraper.app import ScrapingUtils
from src.core.scraper.processor import ImagesProcessor
images_processor = ImagesProcessor()
scraper_utils = ScrapingUtils()

In [4]:
url = "https://www.auteco.com.co/moto-tvs-sport-100-kls/p"

actions = [
    {"type": "scroll", "direction": "down"},
    {"type": "wait", "milliseconds": 2000},
    {"type": "scroll", "direction": "down"},
    # {"type": "wait", "milliseconds": 2000},
    # {"type": "scroll", "direction": "down"},
    # {"type": "wait", "milliseconds": 2000},
    # {"type": "scroll", "direction": "down"},
]

result = images_processor.test_extract(url,
                formats=["html"],
                actions=actions,
                wait_for=1200)
# print(result.html)
# model_data = images_processor.get_model_data(url=url)

In [6]:
print(result.html)

<!DOCTYPE html><html lang="es-CO">
<body class="bg-base">
  <div id="styles_iconpack" style="display:none"><svg class="dn" height="0" version="1.1" width="0" xmlns="http://www.w3.org/2000/svg"><defs><g id="bnd-logo"><path d="M14.8018 2H2.8541C1.92768 2 1.33445 2.93596 1.76795 3.71405L2.96368 5.86466H0.796482C0.659276 5.8647 0.524407 5.89841 0.404937 5.96253C0.285467 6.02665 0.185446 6.119 0.114562 6.23064C0.0436777 6.34228 0.0043325 6.46943 0.000337815 6.59976C-0.00365688 6.73009 0.0278339 6.8592 0.0917605 6.97457L3.93578 13.8888C4.00355 14.0104 4.10491 14.1121 4.22896 14.1831C4.35301 14.254 4.49507 14.2915 4.63987 14.2915C4.78466 14.2915 4.92672 14.254 5.05077 14.1831C5.17483 14.1121 5.27618 14.0104 5.34395 13.8888L6.38793 12.0211L7.69771 14.3775C8.15868 15.2063 9.40744 15.2078 9.87001 14.38L15.8585 3.67064C16.2818 2.91319 15.7043 2 14.8018 2ZM9.43491 6.57566L6.85274 11.1944C6.80761 11.2753 6.74016 11.3429 6.65762 11.3901C6.57508 11.4373 6.48058 11.4622 6.38426 11.4622C6.28794 11.4622

In [86]:
import re
from bs4 import BeautifulSoup

# 1) HTML base
base = images_processor.test_extract(
    url,
    formats=["html"],
    wait_for=1200
)

soup = BeautifulSoup(base.html, "html.parser")

# 2) Detectar clases únicas tipo dsm_shapes_N
shape_classes = []
for el in soup.select("div.dsm_shapes"):
    classes = el.get("class", [])
    for c in classes:
        if re.match(r"^dsm_shapes_\d+$", c):
            shape_classes.append(c)

# quitar duplicados y ordenar por índice numérico
shape_classes = sorted(set(shape_classes), key=lambda x: int(x.split("_")[-1]))
print("Botones detectados:", shape_classes)

Botones detectados: ['dsm_shapes_0', 'dsm_shapes_1', 'dsm_shapes_2', 'dsm_shapes_3']


In [87]:
# 3) Un request por botón (simple, sin mezclar clicks)
html_by_shape = {}

for shape_cls in shape_classes:
    selector = f"div.dsm_shapes.{shape_cls}"
    result_click = images_processor.test_extract(
        url,
        formats=["html"],
        actions=[
            {"type": "click", "selector": selector},
            {"type": "wait", "milliseconds": 1200}
        ],
        wait_for=1200
    )
    html_by_shape[shape_cls] = result_click.html
    print(f"OK -> {shape_cls}")



OK -> dsm_shapes_0
OK -> dsm_shapes_1
OK -> dsm_shapes_2
OK -> dsm_shapes_3


In [88]:
html_by_shape

{'dsm_shapes_0': '<!DOCTYPE html><html lang="es" class="js"><body class="wp-singular motos-template-default single single-motos postid-987501764 wp-theme-Divi et-tb-has-template et-tb-has-header et-tb-has-body et-tb-has-footer wp-schema-pro-2.7.3 cookies-not-set et_pb_button_helper_class et_cover_background et_pb_gutter linux et_pb_gutters3 et_pb_pagebuilder_layout et_divi_theme et-db dm-custom-header dm-bm-pos-right dm-off-canvas dm-side-slide dm-circle-expand dm-menuside-right dm-ch-logo-pos-left dm-ch-cart-icon-pos-none dm-ch-search-icon-pos-none dm-ch-account-icon-pos-none dm-fixed-scroll collapse-submenu chrome dmm-fixed dm-fixed-header" style="overflow-x: hidden;" data-scroll-step="1762" data-type="terms-conditions" data-theme="upcoming"><div id="et-pb-motion-effects-offset-tracker"></div><div class="overlay-loader" style="display: none;">\n\t\t\t\t\t\t\t<div class="loader" style="display: none;">\n\t\t\t\t\t\t\t\t<div class="lds-css ng-scope"><div style="width:100%;height:100%" 

In [ ]:
# 4) Extraer image-rotator por cada estado
for shape_cls, html in html_by_shape.items():
    soup_click = BeautifulSoup(html, "html.parser")
    rotators = soup_click.find_all("image-rotator")
    print(shape_cls, "rotators:", len(rotators))
    # aqui luego extraes src/srcset según necesites

# El div padre correcto es el que tiene la clase con "spec-productSpecificationGroup pb0"
rotators = soup.find_all("image-rotator")
rotator_data = []
for rot in rotators:
    src = rot.get("src")
    total_images = rot.get("total-images")
    rotator_data.append({
        "src": src,
        "total_images": total_images
    })

In [82]:
rotator_data

[{'src': '/wp-content/uploads/sites/2/2025/03/grupouma-pulsar-ns200-fi-abs-grey-${var}.webp',
  'total_images': '16'},
 {'src': '/wp-content/uploads/sites/2/2025/03/grupouma-pulsar-ns200-fi-abs-black-${var}.webp',
  'total_images': '16'},
 {'src': '/wp-content/uploads/sites/2/2025/05/grupouma-pulsar-ns-200-fi-abs-azul-azalea-${var}.webp',
  'total_images': '8'},
 {'src': '/wp-content/uploads/sites/2/2025/10/grupouma-pulsar-ns200-fi-abs-dc-roja-${var}.webp',
  'total_images': '16'}]

In [ ]:
# # Bajaj CO
# from bs4 import BeautifulSoup

# html = result.html

# soup = BeautifulSoup(html, "html.parser")
# # El div padre correcto es el que tiene la clase con "spec-productSpecificationGroup pb0"
# specs_divs = soup.find_all("div", class_="dsm_shapes")
# count_dsm_shapes = len(specs_divs)
# print(f"Número de veces que aparece 'dsm_shapes': {count_dsm_shapes}")

Número de veces que aparece 'dsm_shapes': 4


In [ ]:
result = images_processor.test_extract(url,
                formats=["images"],
                actions=actions,
                wait_for=1200)

In [52]:
print(result.metadata)

title='PULSAR NS200 FI ABS DC | Bajaj | Grupo UMA COL' description='Domina las calles con estilo y seguridad. La Pulsar NS200 FI ABS DC de Bajaj, es la moto urbana definitiva para adrenalina y control total' url='https://grupouma.com/colombia/motos/pulsar/pulsar-ns200-fi-abs-dc/' language='es' keywords=None robots='index, follow, max-image-preview:large, max-snippet:-1, max-video-preview:-1' og_title='Pulsar NS200 FI ABS DC' og_description='Domina las calles con estilo y seguridad. La Pulsar NS200 FI ABS DC de Bajaj, es la moto urbana definitiva para adrenalina y control total' og_url='https://grupouma.com/colombia/motos/pulsar/pulsar-ns200-fi-abs-dc/' og_image='https://grupouma.com/colombia/wp-content/uploads/sites/2/2022/08/grupouma-pulsar-ns200-fi-abs-2026-miniatura-1.webp' og_audio=None og_determiner=None og_locale='es_ES' og_locale_alternate=None og_site_name='UMA Colombia' og_video=None favicon='https://grupouma.com/colombia/wp-content/uploads/sites/2/2021/07/favicon.png' dc_term

In [53]:
for imagen in result.images:
    if ("grupo-uma" in imagen and not any(word in imagen for word in ["logo", "miniatura"])):
        print(imagen)


https://grupouma.com/colombia/wp-content/uploads/sites/2/2025/10/grupo-uma-comparador-pulsar-n160-dc-gi.webp
https://grupouma.com/colombia/wp-content/uploads/sites/2/2026/03/grupo-uma-banner-blog-pulsar-ns400-1024x683.webp
https://grupouma.com/colombia/wp-content/uploads/sites/2/2025/03/grupo-uma-pulsar-ns-fi-abs-200-2026-antes-despues.webp
https://grupouma.com/colombia/wp-content/uploads/sites/2/2025/03/grupo-uma-pulsar-ns-200-fi-abs-2026-azul-form.webp
https://grupouma.com/colombia/wp-content/uploads/sites/2/2025/03/grupo-uma-pulsar-ns-200-fi-abs-2026-azul-01.webp
https://grupouma.com/colombia/wp-content/uploads/sites/2/2025/03/grupo-uma-pulsar-ns-fi-abs-200-2026-antes-despues-bn.webp
https://grupouma.com/colombia/wp-content/uploads/sites/2/2025/03/grupo-uma-mm-boxer-ct-125.webp
https://grupouma.com/colombia/wp-content/uploads/sites/2/2025/03/grupo-uma-pulsar-ns-fi-abs-200-2026-barras-invertidas.webp
https://grupouma.com/colombia/wp-content/uploads/sites/2/2025/03/grupo-uma-pulsar-ns

In [57]:
# Bajaj CO

from bs4 import BeautifulSoup

html = result.html

soup = BeautifulSoup(html, "html.parser")
# El div padre correcto es el que tiene la clase con "spec-productSpecificationGroup pb0"
specs_div = soup.find("image-rotator")
total_images = specs_div.get("total-images")

image_src = specs_div.get("src").replace("-${var}", "").replace(".webp", "")
extension = specs_div.get("src").split(".")[-1]
print(image_src)
print(extension)

url_base = "https://grupouma.com/"
for i in range(1, int(total_images)):
    if i < 10:
        print(f"{url_base}{image_src}-0{i}.{extension}")
    else:
        print(f"{url_base}{image_src}-{i}.{extension}")

/wp-content/uploads/sites/2/2025/03/grupouma-pulsar-ns200-fi-abs-grey
webp
https://grupouma.com//wp-content/uploads/sites/2/2025/03/grupouma-pulsar-ns200-fi-abs-grey-01.webp
https://grupouma.com//wp-content/uploads/sites/2/2025/03/grupouma-pulsar-ns200-fi-abs-grey-02.webp
https://grupouma.com//wp-content/uploads/sites/2/2025/03/grupouma-pulsar-ns200-fi-abs-grey-03.webp
https://grupouma.com//wp-content/uploads/sites/2/2025/03/grupouma-pulsar-ns200-fi-abs-grey-04.webp
https://grupouma.com//wp-content/uploads/sites/2/2025/03/grupouma-pulsar-ns200-fi-abs-grey-05.webp
https://grupouma.com//wp-content/uploads/sites/2/2025/03/grupouma-pulsar-ns200-fi-abs-grey-06.webp
https://grupouma.com//wp-content/uploads/sites/2/2025/03/grupouma-pulsar-ns200-fi-abs-grey-07.webp
https://grupouma.com//wp-content/uploads/sites/2/2025/03/grupouma-pulsar-ns200-fi-abs-grey-08.webp
https://grupouma.com//wp-content/uploads/sites/2/2025/03/grupouma-pulsar-ns200-fi-abs-grey-09.webp
https://grupouma.com//wp-content/u

In [ ]:
# from src.core.scraper.utils import download_images
# download_images(["https://grupouma.com/colombia/wp-content/uploads/sites/2/2025/09/grupo-uma-dominar-interna-volcano-asiento.webp"],
#  "Bajaj Dominar Volcano",
#   f"../src/data/images/Bajaj_Dominar_Volcano_400")

### Manejo con Bs4

In [7]:
html = result.html
print(html)

<!DOCTYPE html><html lang="es" class="js"><body class="wp-singular motos-template-default single single-motos postid-987506604 wp-theme-Divi et-tb-has-template et-tb-has-header et-tb-has-body et-tb-has-footer wp-schema-pro-2.7.3 cookies-not-set et_pb_button_helper_class et_cover_background et_pb_gutter windows et_pb_gutters3 et_pb_pagebuilder_layout et_divi_theme et-db dm-custom-header dm-bm-pos-right dm-off-canvas dm-side-slide dm-circle-expand dm-menuside-right dm-ch-logo-pos-left dm-ch-cart-icon-pos-none dm-ch-search-icon-pos-none dm-ch-account-icon-pos-none dm-fixed-scroll collapse-submenu chrome dmm-fixed" data-scroll-step="0" style="overflow-x: hidden;" data-type="terms-conditions" data-theme="upcoming"><div id="et-pb-motion-effects-offset-tracker"></div><div class="overlay-loader" style="display: none;">
							<div class="loader" style="display: none;">
								<div class="lds-css ng-scope"><div style="width:100%;height:100%" class="lds-rolling"><div></div></div></div>
							<

In [ ]:


# Extraer el valor de total-images del tag image-rotator
image_rotator = specs_div.find("image-galeria") if specs_div else None
total_images = image_rotator.get("total-images") if image_rotator else None # 8
uri_pattern = image_rotator.get("src") if image_rotator else None # /wp-content/uploads/2026/01/akt-jet-evo-negro-01.webp

In [67]:
# Apartir de acá separaremos texto y encontraremos patrones. Lo que buscamos es poder quitar el consecutivo final:
# url original: 'akt-jet-evo-negro-01.webp'
# objetivo:
#   - uri base: /wp-content/uploads/2026/01/
#   - model_name_uri: akt-jet-evo-negro
#   - extension: webp
# url objetivo: https://aktmotos.com/{url_base}/{model_name_uri}/0{i}.{extension}
text_to_extract_extension = uri_pattern.split("/")[-1] # 'akt-jet-evo-negro-01.webp'

In [68]:
uri_base = uri_pattern.replace(text_to_extract_extension, "") # /wp-content/uploads/2026/01/ <-- Apartir de acá armaremos la URL
model_name_uri = text_to_extract_extension.split(".")[0] # akt-jet-evo-negro-01 <-- Se tiene parte del nombre de la imagen
model_name = model_name_uri.replace("-01", "") # akt-jet-evo-negro <-- Se quita el consecutivo final # ! Posiblemente no todos inicien con -01
extension = text_to_extract_extension.split(".")[-1] # webp <-- Usada para completar la extensión de la URL

In [69]:
uri_base

'/wp-content/uploads/2026/01/'

In [70]:
model_name

'akt-jet-evo-negro'

In [71]:
extension

'webp'

In [75]:
total_images

'8'

In [80]:

image_list = []
for i in range(0,int(total_images)):
    image_list.append(f"https://aktmotos.com/{uri_base}{model_name}-0{i+1}.{extension}")

In [81]:
image_list

['https://aktmotos.com//wp-content/uploads/2026/01/akt-jet-evo-negro-01.webp',
 'https://aktmotos.com//wp-content/uploads/2026/01/akt-jet-evo-negro-02.webp',
 'https://aktmotos.com//wp-content/uploads/2026/01/akt-jet-evo-negro-03.webp',
 'https://aktmotos.com//wp-content/uploads/2026/01/akt-jet-evo-negro-04.webp',
 'https://aktmotos.com//wp-content/uploads/2026/01/akt-jet-evo-negro-05.webp',
 'https://aktmotos.com//wp-content/uploads/2026/01/akt-jet-evo-negro-06.webp',
 'https://aktmotos.com//wp-content/uploads/2026/01/akt-jet-evo-negro-07.webp',
 'https://aktmotos.com//wp-content/uploads/2026/01/akt-jet-evo-negro-08.webp']

In [7]:
for image in result.images:
    # Con esta se arma la url base
    if "interna-de-producto" in image:
        url_base_to_process = image # 'https://media.autecomobility.com/recursos/marcas/tvs/raider-125/interna-de-producto/Imagen_Fondo_Texto_detalle_2_TVS.webp'
        break # Solo con la primera coincidencia sirve

# Se separa por barras (/) y se elimina el último elemento para crear así la url base
text_to_eliminate = url_base_to_process.split("/")[-1] # /interna-de-producto/'
url_base = url_base_to_process.replace(text_to_eliminate, "") # 'https://media.autecomobility.com/recursos/marcas/tvs/raider-125/interna-de-producto/'

TypeError: 'NoneType' object is not iterable

In [ ]:
url_base_list = []
for url in range(0,7):
    url_base_list.append(f"{url_base}/Galeria-imagen-{url+1}")

url_base_list

In [ ]:
import requests
default_extension = "webp"
alt_extension = "png"
url_list_checked = []
# Itera entre cada url sin extensión y agrega la extensión por defecto
for url in url_base_list:
    url_to_check = f"{url}.{default_extension}"
    response = requests.get(url_to_check) # Se hace un reuquest para comprobar el status_code que retorne
    if response.status_code == 404: # Si el status_code es 404, se agrega la extensión alternativa
        response = requests.get(f"{url}.{alt_extension}") # Se vuelve a hacer el request con la extensión alternativa
        if response.status_code == 200: # Si el status_code es 200, se agrega la url con la extensión alternativa
            url_list_checked.append(f"{url}.{alt_extension}")
    if response.status_code == 200: # Si el status_code es 200, se agrega la url con la extensión alternativa
        url_list_checked.append(f"{url}.{alt_extension}")

In [ ]:
url_list_checked

In [ ]:
# import requests
# default_extension = "webp"
# alt_extension = "png"
# url_list_checked = []
# # Itera entre cada url sin extensión y agrega la extensión por defecto
# for url in url_base_list:
#     print(f"Probando con extensión por defecto: {url}.{default_extension}")
#     response = requests.get(f"{url}.{default_extension}") # Se hace un reuquest para comprobar el status_code que retorne
#     if response.status_code == 404: # Si el status_code es 404, se agrega la extensión alternativa
#         print(f"Probando con extensión alternativa: {url}.{alt_extension}")
#         response = requests.get(f"{url}.{alt_extension}") # Se vuelve a hacer el request con la extensión alternativa
#         if response.status_code == 200: # Si el status_code es 200, se agrega la url con la extensión alternativa
#             url_list_checked.append(f"{url}.{alt_extension}")
#     url_list_checked.append(f"{url}.{alt_extension}")


In [ ]:
url_list_checked

In [ ]:
result.images

In [ ]:

#
urls_images_list = []
for image in result.images:
    if "width=800" in image:
        urls_images_list.append(image)